<a href="https://colab.research.google.com/github/zahere-dev/advanced-rag/blob/main/01-chunking/recursive_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recursive Chunking

Part of the RAG Master Class: https://www.youtube.com/watch?v=hEG3-L4l5R4

## Install required packages

In [ ]:
!pip install langchain langchain_community langchain-experimental langchain-openai chromadb datasets ragas

In [3]:
import bs4
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

## Load the dataset

In [ ]:
loader = TextLoader("/content/medical_case_studies_synthetica_dataset.txt")
documents = loader.load()
documents

In [6]:
chunk_size = 1000
chunk_overlap = 200
separators = ["\n\n", "\n"]  # Split on double and single newlines

# Create text splitter object
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=separators)

recursive_chunks = text_splitter.create_documents([d.page_content for d in documents])


for i in range(50):
  print("Start_of_new_chunk"+ "*-*-"*50)
  print(recursive_chunks[i])
  print("End_of_chunk"+ "*-*-"*50)
  print("\n")

Start_of_new_chunk*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
page_content='## Case Study 1: Novel Treatment for Type 2 Diabetes

Patient: 52-year-old male, diagnosed with Type 2 Diabetes 8 years ago
Current Treatment: Metformin (1000mg twice daily), Lifestyle modifications

Background:
John Doe, a 52-year-old male, has been managing Type 2 Diabetes for the past 8 years with oral medication and lifestyle changes. Despite adherence to his treatment plan, his glycemic control has been suboptimal in recent months. His most recent HbA1c was 8.2%, indicating poor glucose management.

Intervention:
A novel GLP-1 receptor agonist/glucose-dependent insulinotropic polypeptide (GIP) dual agonist, Tirzepatide, was introduced to John's treatment regimen. The medication was administered once weekly via subcutaneous injection, starting at 2.5mg 

In [7]:
import os
from langchain_openai.embeddings import OpenAIEmbeddings
from google.colab import userdata
from langchain_community.vectorstores import Chroma

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] =  userdata.get("OPENAI_API_KEY")  # Add your OpenAI API Key


embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
recursive_chunk_vectorstore = Chroma.from_documents(recursive_chunks, embedding=embeddings)

In [8]:
recursive_chunk_retriever = recursive_chunk_vectorstore.as_retriever(search_kwargs={"k" : 1})
recursive_chunk_retriever.invoke("How are nanoparticles being used in Alzheimer's disease treatment??")

[Document(metadata={}, page_content="Discussion:\nThis study demonstrates the potential of nanotechnology-based approaches for the treatment of Alzheimer's Disease. The designed nanoparticles successfully crossed the blood-brain barrier and delivered functional Aβ-degrading enzyme to the brain. The significant reduction in Aβ levels, coupled with improved cognitive function and reduced neuroinflammation, suggests a promising therapeutic strategy.\n\nThe superior performance of the enzyme-loaded nanoparticles compared to free enzyme highlights the importance of targeted delivery in overcoming the blood-brain barrier. The observed cognitive improvements provide hope for not just slowing disease progression, but potentially reversing some cognitive deficits.")]

In [9]:
from langchain_core.prompts import ChatPromptTemplate

rag_template = """\
Use the following context to answer the user's query. If you cannot answer, please respond with 'I don't know'.

User's Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

recursive_rag_chain = (
    {"context" : recursive_chunk_retriever, "question" : RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [11]:
synthetic_data_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False
)
#
synthetic_data_chunks = synthetic_data_splitter.create_documents([d.page_content for d in documents])
print(len(synthetic_data_chunks))

69


In [12]:
questions = []
ground_truths_recursive = []
contexts = []
answers = []

question_prompt = """\
You are a teacher preparing a test. Please create a question that can be answered by referencing the following context.

Context:
{context}
"""

question_prompt = ChatPromptTemplate.from_template(question_prompt)

ground_truth_prompt = """\
Use the following context and question to answer this question using *only* the provided context.

Question:
{question}

Context:
{context}
"""

ground_truth_prompt = ChatPromptTemplate.from_template(ground_truth_prompt)

question_chain = question_prompt | llm | StrOutputParser()
ground_truth_chain = ground_truth_prompt | llm | StrOutputParser()

for chunk in synthetic_data_chunks[10:20]:
  questions.append(question_chain.invoke({"context" : chunk.page_content}))
  contexts.append([chunk.page_content])
  ground_truths_recursive.append(ground_truth_chain.invoke({"question" : questions[-1], "context" : contexts[-1]}))
  answers.append(recursive_rag_chain.invoke(questions[-1]))

In [13]:
from datasets import load_dataset, Dataset

qagc_list = []

for question, answer, context, ground_truth in zip(questions, answers, contexts, ground_truths_recursive):
  qagc_list.append({
      "question" : question,
      "answer" : answer,
      "contexts" : context,
      "ground_truth" : ground_truth
  })

eval_dataset = Dataset.from_list(qagc_list)
eval_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 10
})

In [14]:
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
)

from ragas import evaluate

result = evaluate(
    eval_dataset,
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
        context_recall,
    ],
)
result

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in _VertexAIBase has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in _VertexAICommon has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/ragas/metrics/__init__.py:4: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_precision': 1.0000, 'faithfulness': 0.8276, 'answer_relevancy': 0.9167, 'context_recall': 0.9056}

In [13]:
results_df = result.to_pandas()
results_df

,question,contexts,answer,ground_truth,context_precision,faithfulness,answer_relevancy,context_recall
0,**Question:**\n\nBased on the provided context...,"[Patient: 58-year-old male, newly diagnosed Gl...",The personalized neoantigen vaccine interventi...,The personalized neoantigen vaccine interventi...,1.0,0.636364,0.951823,1.000000
1,**Question:**\n\nBased on the provided context...,[Results:\nImmediate Post-Treatment:\n1. Vacci...,"At the 24-month follow-up, the notable outcome...","At the 24-month follow-up, the notable outcome...",1.0,1.000000,0.862945,1.000000
2,**Question:**\n\nBased on the provided context...,[The combination of the neoantigen vaccine wit...,The potential benefits of combining a neoantig...,The potential benefits of combining a neoantig...,1.0,1.000000,0.948816,1.000000
3,**Question:**\n\nBased on the research summari...,[## Research Summary: Nanotechnology-Based Dru...,The key components of the nanoparticles design...,The key components of the nanoparticles design...,1.0,0.538462,0.890427,1.000000
4,**Question:**\n\nBased on the in vivo study de...,[3. In Vivo Studies:\n - Animal model: APP/P...,The treatment group expected to show the most ...,"Based on the in vivo study described, the trea...",1.0,0.500000,0.877520,0.750000
5,**Question:** Based on the provided context re...,[2. In Vivo:\n a) Brain Aβ Levels:\n - ...,**Key Findings:**\n\n1. **Brain Aβ Levels:**\n...,**Summary of Key Findings:**\n\n1. **Brain Aβ ...,1.0,1.000000,0.904245,1.000000
6,"**Question:**\n\nBased on the study discussed,...",[Discussion:\nThis study demonstrates the pote...,The key advantages of using nanotechnology-bas...,The key advantages of using nanotechnology-bas...,1.0,0.750000,0.928764,1.000000
7,**Question:**\n\nBased on the provided context...,"[The favorable safety profile is encouraging, ...",The main limitations identified in the preclin...,The main limitations identified in the preclin...,1.0,0.250000,0.940922,0.666667
8,**Question:**\n\nBased on the case study of Sa...,[## Case Study 4: CAR-T Cell Therapy for Refra...,Sarah Thompson underwent multiple lines of the...,Sarah Thompson underwent multiple lines of the...,1.0,1.000000,0.893110,1.000000
9,**Question:**\n\nBased on Sarah's experience i...,[Intervention:\nSarah was enrolled in a clinic...,B) Achievement of stringent complete response ...,The significant outcomes observed at the 3-mon...,1.0,1.000000,0.835022,1.000000


In [14]:
results_df.head(1)["question"][0]

'**Question:**\n\nBased on the provided context, describe the steps involved in the personalized neoantigen vaccine intervention that Michael Johnson underwent as part of his treatment for Glioblastoma Multiforme. Include at least three specific steps in your answer.'